# Convergent Validity of Technical Metrics (Dataset B)

## Purpose
This notebook validates the three technical DeepEval metrics in the IHAEF framework — Answer Relevancy (AR), Contextual Relevancy (CR), and Faithfulness (F) — by measuring their agreement with single-annotator human labels on Dataset B (n=21).

## Approach
For each metric, we compare DeepEval's continuous score against the binary human label. Cases marked N/A by the human annotator (where the bot's response did not require retrieval and thus CR/F do not apply) are excluded from CR and F analyses.

Validity is assessed via Spearman rank correlation between metric scores and human labels, with 95% bootstrap confidence intervals. Where label distributions are too imbalanced for stable correlation estimation, descriptive analysis is reported instead.

## Hypothesis framing
Per metric:
- **H₀**: The metric does not converge with human judgment (ρ ≤ 0; no positive association)
- **H₁**: The metric converges with human judgment (ρ > 0; positive association)

Reject H₀ if ρ > 0 with 95% CI excluding zero and p < 0.05.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

eval_df = pd.read_csv('../data/evaluation_results.csv')
ann_df = pd.read_csv('../data/qa_pairs_annotated.csv')

print(f'Evaluation results: {len(eval_df)} rows, columns: {list(eval_df.columns)[:6]}...')
print(f'Annotations: {len(ann_df)} rows, columns: {list(ann_df.columns)}')

Evaluation results: 21 rows, columns: ['conv_id', 'user_input', 'bot_output', 'retrieval_context', 'Answer Relevancy', 'Answer Relevancy Reason']...
Annotations: 21 rows, columns: ['user_input', 'bot_output', 'context', 'answer_relevancy_label', 'context_relevancy_label', 'faithfulness_label']


The two files are not in the same row order, so we merge on `user_input` rather than relying on index alignment.

In [2]:
merged = eval_df.merge(
    ann_df[['user_input', 'answer_relevancy_label',
            'context_relevancy_label', 'faithfulness_label']],
    on='user_input',
    how='inner'
)
print(f'Merged rows: {len(merged)} (expected 21)')
print()
for col in ['answer_relevancy_label', 'context_relevancy_label', 'faithfulness_label']:
    print(f'{col}: {merged[col].value_counts(dropna=False).to_dict()}')

Merged rows: 21 (expected 21)

answer_relevancy_label: {1: 20, 0: 1}
context_relevancy_label: {0.0: 11, nan: 8, 1.0: 2}
faithfulness_label: {1.0: 12, nan: 8, 0.0: 1}


**Label distributions:**

- AR: 20 positive, 1 negative
- CR: 11 negative, 2 positive, 8 N/A
- F: 12 positive, 1 negative, 8 N/A

The 8 N/A rows in CR and F correspond to cases where the chatbot's response did not require retrieved context (e.g., clarification requests, expressions of empathy, refusals). Contextual Relevancy and Faithfulness are not applicable to such responses and these rows are excluded from the corresponding analyses.

AR has a heavily skewed distribution (20/1), and CR/F have small effective sample sizes (n=13) after N/A exclusion. These limitations are documented and inform the choice of statistical analysis per metric.

## Spearman correlation with bootstrap CI

In [ ]:
def spearman_with_ci(scores, labels, n_bootstrap=10000):
    """Spearman rho with 95% bootstrap CI.
    
    Skips bootstrap resamples where either variable has no variation
    (correlation undefined).
    """
    n = len(scores)
    rho, p = stats.spearmanr(scores, labels)
    
    boot_rhos = []
    indices = np.arange(n)
    for _ in range(n_bootstrap):
        sample_idx = rng.choice(indices, size=n, replace=True)
        s = scores[sample_idx]
        l = labels[sample_idx]
        if len(np.unique(l)) < 2 or len(np.unique(s)) < 2:
            continue
        r, _ = stats.spearmanr(s, l)
        if not np.isnan(r):
            boot_rhos.append(r)
    
    ci_low = np.percentile(boot_rhos, 2.5)
    ci_high = np.percentile(boot_rhos, 97.5)
    return rho, p, ci_low, ci_high, len(boot_rhos)

configs = [
    ('Answer Relevancy', 'answer_relevancy_label', 'AR'),
    ('Contextual Relevancy', 'context_relevancy_label', 'CR'),
    ('Faithfulness', 'faithfulness_label', 'F'),
]

print(f"{'Metric':<8} {'n':>4} {'rho':>8} {'p':>8} {'95% CI':>22} {'valid boots':>14}")
print('-' * 70)
for score_col, label_col, name in configs:
    sub = merged[[score_col, label_col]].dropna()
    scores = sub[score_col].to_numpy()
    labels = sub[label_col].to_numpy()
    rho, p, lo, hi, b = spearman_with_ci(scores, labels)
    print(f'{name:<8} {len(sub):>4} {rho:>8.3f} {p:>8.3f} [{lo:>6.3f}, {hi:>6.3f}] {b:>14}')

Metric      n      rho        p                 95% CI    valid boots
----------------------------------------------------------------------
AR         21    0.385    0.085 [ 0.374,  0.690]           6276
CR         13    0.570    0.042 [ 0.310,  0.820]           8803
F          13   -0.083    0.787 [-0.234, -0.083]           4144


### Interpretation of Spearman results

**CR**: ρ = 0.570, p = 0.042, 95% CI [0.31, 0.82]. Statistically significant positive correlation with CI excluding zero. This is the strongest formal validity result of the three metrics. **Reject H₀**.

**AR and F**: The bootstrap CIs are unreliable due to extreme class imbalance (20/1 for AR, 12/1 for F after N/A exclusion). Many bootstrap resamples did not contain the single minority case, producing a bimodal distribution of bootstrap correlations and CIs that exclude the original point estimate. This is a known limitation of bootstrap on severely imbalanced data, not an analysis error.

Because Spearman cannot be reliably interpreted for AR and F on this dataset, we use descriptive analysis instead.

## AR: Descriptive analysis

Spearman ρ is uninformative for AR because there is only one negative case and many tied scores at 1.0 among the positives, which mechanically reduce ρ even when class separation is clean. Direct inspection of scores by label is more informative.

In [4]:
ar_data = merged[['Answer Relevancy', 'answer_relevancy_label']].sort_values('answer_relevancy_label')
print('AR scores by human label:')
print(ar_data.to_string())
print()

pos = ar_data[ar_data['answer_relevancy_label'] == 1]['Answer Relevancy']
neg = ar_data[ar_data['answer_relevancy_label'] == 0]['Answer Relevancy']
print(f'Positives (n={len(pos)}): mean={pos.mean():.3f}, min={pos.min():.3f}, max={pos.max():.3f}')
print(f'Negatives (n={len(neg)}): mean={neg.mean():.3f}, min={neg.min():.3f}, max={neg.max():.3f}')
print(f'Separation: lowest positive score ({pos.min():.3f}) vs highest negative score ({neg.max():.3f})')

AR scores by human label:
    Answer Relevancy  answer_relevancy_label
17          0.000000                       0
0           0.750000                       1
2           1.000000                       1
1           1.000000                       1
4           0.923077                       1
5           1.000000                       1
6           1.000000                       1
3           1.000000                       1
8           0.875000                       1
9           0.785714                       1
10          0.800000                       1
11          0.900000                       1
12          0.615385                       1
13          1.000000                       1
14          0.923077                       1
7           1.000000                       1
15          1.000000                       1
16          1.000000                       1
18          0.818182                       1
19          0.941176                       1
20          0.666667         

### Interpretation of AR results

The single case labelled irrelevant by the human annotator received a score of 0.00 from AR. The 20 cases labelled relevant received scores ranging from 0.62 to 1.00. **Class separation is perfect**: there is no overlap between the score assigned to the negative case and the scores assigned to positive cases.

AR's behaviour is consistent with the construct: relevant answers are scored highly, the irrelevant answer is scored at the floor. **Reject H₀**, with the documented limitation that this validity claim rests on a single negative case (n=1 for the minority class).

## F: Descriptive analysis

In [5]:
f_data = merged[['Faithfulness', 'faithfulness_label']].dropna().sort_values('faithfulness_label')
print('F scores by human label (N/A cases excluded):')
print(f_data.to_string())
print()

f_pos = f_data[f_data['faithfulness_label'] == 1]['Faithfulness']
f_neg = f_data[f_data['faithfulness_label'] == 0]['Faithfulness']
print(f'Positives (n={len(f_pos)}): mean={f_pos.mean():.3f}, min={f_pos.min():.3f}, max={f_pos.max():.3f}')
print(f'Negatives (n={len(f_neg)}): mean={f_neg.mean():.3f}, min={f_neg.min():.3f}, max={f_neg.max():.3f}')
print()
print(f'Unique F scores: {sorted(f_data["Faithfulness"].unique())}')
print(f'Proportion of cases scoring exactly 1.0: {(f_data["Faithfulness"] == 1.0).mean():.1%}')

F scores by human label (N/A cases excluded):
    Faithfulness  faithfulness_label
11      1.000000                 0.0
4       1.000000                 1.0
7       1.000000                 1.0
6       1.000000                 1.0
8       1.000000                 1.0
9       1.000000                 1.0
12      1.000000                 1.0
13      1.000000                 1.0
14      0.888889                 1.0
17      1.000000                 1.0
18      1.000000                 1.0
19      1.000000                 1.0
20      1.000000                 1.0

Positives (n=12): mean=0.991, min=0.889, max=1.000
Negatives (n=1): mean=1.000, min=1.000, max=1.000

Unique F scores: [np.float64(0.8888888888888888), np.float64(1.0)]
Proportion of cases scoring exactly 1.0: 92.3%


### Interpretation of F results

Faithfulness returned a score of 1.0 on 12 of 13 evaluable cases, including the single case labelled unfaithful by the human annotator. The score distribution has effectively no variance: 12 cases at 1.0, one case at 0.889, one case at 1.0 with a negative label.

Because F produces no meaningful score variation on this dataset, **no validation method can extract a signal**. **H₀ cannot be rejected**, but the failure to reject is not informative about F's true discriminative ability.

The most plausible explanation is a **scope mismatch**: Faithfulness evaluates whether claims in the response contradict the retrieval context, and HIA's responses in Dataset B consist primarily of clarification requests, expressions of empathy, and referrals to other services — response types that contain few substantive factual claims that could be unfaithful to retrieval. Under this interpretation, F is not malfunctioning; the dataset simply does not exercise the construct F measures.

An alternative explanation — that F is saturated and would return 1.0 regardless of the actual answer-context relationship — cannot be ruled out from this data. Resolving this ambiguity requires evaluating F on a dataset containing substantive grounded responses, which is left to future work.

## Summary

| Metric | Method | Result | Decision |
|--------|--------|--------|----------|
| AR | Descriptive (class separation) | Perfect separation: 0.00 vs [0.62, 1.00] | Reject H₀ (with n=1 minority limitation) |
| CR | Spearman with bootstrap CI | ρ = 0.570, p = 0.042, 95% CI [0.31, 0.82] | Reject H₀ |
| F  | Descriptive (no variance) | 12/13 scores at 1.0; no signal | Inconclusive; scope mismatch likely |

Two of the three technical metrics show evidence of convergent validity with single-annotator human judgment on Dataset B. The third (Faithfulness) could not be validated on this dataset due to a likely scope mismatch between the metric and the chatbot response style. F's applicability should be re-examined on a dataset containing substantive grounded outputs.

These findings replace, rather than supplement, the earlier construct validation work (which suffered from design-level limitations including negative-control label noise and positive-set contamination). The convergent validity approach used here, while smaller in scope, produces interpretable validity evidence grounded in human judgment rather than synthetic shuffling.